In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://cais.gsi.go.jp/YOCHIREN/"
REPORT_LIST = urljoin(BASE_URL, "report.html")
CRAWL_DELAY = 2
UA = "research-bot/0.1 (+@example.com)"

KEYWARD = "東海・紀伊半島・四国における短期的スロースリップイベント"
KEYWARD2 = "西南日本における短期的スロースリップイベント"
KEYWARD3 = "西南日本の短期的スロースリップ活動"
KEYWARD4 = "西南日本における短期的スロースリップ- イベント"

OUTDIR = Path("../downloads")
OUTDIR.mkdir(exist_ok=True)
YEAR_OK = set(range(2007, 2026))


def yymm_int(y: int, m: int) -> int:
    return y * 100 + m


def report_period_from_publish_date(year: int, month: int):
    if month == 3:
        return (year - 1, 5), (year - 1, 10)
    if month == 9:
        return (year - 1, 11), (year, 4)
    return None, None


def find_block_for(year: int, month: int, volumes: list[dict]):
    ym = yymm_int(year, month)
    for b in volumes:
        s_ym = yymm_int(b["start"][0], b["start"][1])
        e_ym = yymm_int(b["end"][0], b["end"][1]) if b["end"] else None
        if e_ym is None:
            if ym >= s_ym:
                return b
        else:
            if s_ym <= ym <= e_ym:
                return b
    return None


def get_volumes_data(session: requests.Session) -> list[dict]:
    r = session.get(REPORT_LIST, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    range_re = re.compile(
        r"（\s*(\d{4})年(\d{1,2})月\s*〜\s*(?:(\d{4})年(\d{1,2})月)?\s*）"
    )
    publish_re = re.compile(r"（\s*(\d{4})年(\d{1,2})月\s*）")
    vols = []
    for a in soup.select('a[href*="report.html?id="]'):
        href = a.get("href")
        parent_text = a.parent.get_text(" ", strip=True)
        m = range_re.search(parent_text)
        if not m:
            continue
        y1, m1, y2, m2 = m.groups()
        start = (int(y1), int(m1))
        end = (int(y2), int(m2)) if y2 and m2 else None

        vols.append(
            {
                "title": a.get_text(strip=True),
                "start": start,
                "end": end,
                "href": href,
            }
        )

    for a in soup.select('a[href^="report/index"]'):
        href = a.get("href")
        parent_text = a.parent.get_text(" ", strip=True)
        m = publish_re.search(parent_text)
        if not m:
            continue
        start, end = report_period_from_publish_date(int(m.group(1)), int(m.group(2)))
        if start is None:
            continue
        vols.append(
            {
                "title": a.get_text(strip=True),
                "start": start,
                "end": end,
                "href": href,
            }
        )
    return vols


def list_index_links_in_volume(
    session: requests.Session, volume_href: str, seen_index: set
) -> list[str]:
    url = urljoin(BASE_URL, volume_href)
    if re.match(r"^report/index\d+\.html$", volume_href):
        if volume_href in seen_index:
            return [], seen_index
        seen_index.add(volume_href)
        return [volume_href], seen_index

    r = session.get(url, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")
    index_re = re.compile(r"^report/index\d+\.html$")
    range_re = re.compile(r"（\s*(\d{4})年(\d{1,2})月\s*）")

    hrefs = []
    for a in soup.find_all("a", href=index_re):
        href_raw = a.get("href")
        if href_raw in seen_index:
            continue

        dt = a.find_parent("dt")
        text_block = dt.get_text(" ", strip=True)
        m = range_re.search(text_block)
        if not m:
            continue

        start, end = report_period_from_publish_date(int(m.group(1)), int(m.group(2)))
        if start is None:
            year = int(m.group(1))
            if year not in YEAR_OK:
                continue
        elif not any(
            yymm_int(start[0], start[1])
            <= yymm_int(year, month)
            <= yymm_int(end[0], end[1])
            for year in YEAR_OK
            for month in range(1, 13)
        ):
            continue

        hrefs.append(href_raw)
        seen_index.add(href_raw)

    return hrefs, seen_index


def download_matching_pdfs(
    session: requests.Session,
    index_href: str,
    keyword: str,
    keyword2: str,
    keyword3: str,
    keyword4: str,
):
    index_url = urljoin(BASE_URL, index_href)
    print(f"[INDEX] {index_url}")
    r = session.get(index_url, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    primary_hits = []
    secondary_hits = []
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True)
        href = a["href"]

        if not href.lower().endswith(".pdf"):
            continue

        pdf_url = urljoin(index_url, href)

        if keyword in text:
            fname = f"{text.replace(keyword, '').strip() or text}.pdf"
            primary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword2 in text:
            fname = f"{text.replace(keyword2, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword3 in text:
            fname = f"{text.replace(keyword3, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))
        elif keyword4 in text:
            fname = f"{text.replace(keyword4, '').strip() or text}.pdf"
            secondary_hits.append((pdf_url, OUTDIR / fname))

    hits = primary_hits if primary_hits else secondary_hits

    for pdf_url, outpath in hits:
        try:
            print(f"[PDF]   {pdf_url}")
            resp = session.get(pdf_url, timeout=60)
            resp.encoding = resp.apparent_encoding
            resp.raise_for_status()
            outpath.write_bytes(resp.content)
            print(f"[SAVE]  {outpath}")
        except Exception as e:
            print(f"[WARN]  {pdf_url} -> {e}")
        time.sleep(CRAWL_DELAY)


with requests.Session() as session:
    session.headers.update({"User-Agent": UA})

    volumes = get_volumes_data(session)

    target_volume_hrefs = []
    seen_href = set()
    seen_index = set()

    for year in YEAR_OK:
        for month in range(1, 13):
            b = find_block_for(year, month, volumes)
            if not b:
                continue
            href = b["href"]
            if href not in seen_href:
                seen_href.add(href)
                target_volume_hrefs.append(href)

    for vhref in target_volume_hrefs:
        index_list, seen_index = list_index_links_in_volume(session, vhref, seen_index)

        for ihref in index_list:
            download_matching_pdfs(
                session, ihref, KEYWARD, KEYWARD2, KEYWARD3, KEYWARD4
            )
            time.sleep(CRAWL_DELAY)

[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index115.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou115/05_03.pdf
[SAVE]  downloads/（2025年5月〜2025年10月）（産総研・防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index80.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou80/08_02.pdf
[SAVE]  downloads/（2007年11月〜2008年3月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index79.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou79/09_03.pdf
[SAVE]  downloads/（2007年5-10月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index78.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou78/08_04.pdf
[SAVE]  downloads/（2007年2月〜2007年4月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index77.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou77/7-3.pdf
[SAVE]  downloads/（2006年6月〜11月）（防災科研）.pdf
[INDEX] https://cais.gsi.go.jp/YOCHIREN/report/index90.html
[PDF]   https://cais.gsi.go.jp/YOCHIREN/report/kaihou90/06_02.pdf
[SAVE]  downloads